In [ ]:
# Data: ../csv/Fig2_H_entropy/  (the original loading code is kept below each data line, as comments)
import os, glob, pickle, re, sys   # standard imports
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'analysis_scripts'))   # the criticality library
import pandas as pd
import numpy as np
CSV = Path('..') / 'csv' / 'Fig2_H_entropy'
OUT = Path('..') / 'figure_output'
OUT.mkdir(exist_ok=True)


def read_csv(name, **kw):
    """A table of this notebook (round_trip: the floats exactly as written)."""
    return pd.read_csv(CSV / name, float_precision='round_trip', **kw)

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
from scipy.stats import sem
import os
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import sem as scipy_sem # Use a named import to avoid confusion
from scipy.stats import linregress

import numpy as np
from scipy.stats import gaussian_kde
import matplotlib as mpl

import matplotlib as mpl

# Global font: prefer Arial, with safe fallbacks
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]

# Mathtext to match
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'   # or 'Helvetica'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
def p_to_stars(p):
    p = float(p)
    if not np.isfinite(p):
        return ""
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return "ns"
def _force_sextiles(series, k=6):
    """
    Return Int64 tiles 1..k. Try qcut; if it yields <k bins, fall back to
    rank-based qcut that still returns k bins when N>=k. If N<k -> all NA.
    """
    s = pd.to_numeric(series, errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan)

    # 1) Try ordinary quantile sextiles
    try:
        t = pd.qcut(s, q=k, labels=False, duplicates="drop")
        if pd.Series(t).nunique(dropna=True) == k:
            return t.astype("Int64") + 1
    except Exception:
        pass

    # 2) Fallback: rank-based sextiles (equal-count by rank)
    rk = s.rank(method="first")
    if rk.notna().sum() >= k:
        t = pd.qcut(rk, q=k, labels=False)
        return t.astype("Int64") + 1

    # 3) Not enough rows to form k bins
    return pd.Series(pd.NA, index=s.index, dtype="Int64")
def load_pool_rate_binned_isi(pkl_dir, rate_col, state="NREM", n_bins=6, min_bins_required=3):
    pooled = []
    n_files_used = 0
    n_files_skipped = 0

    for file in os.listdir(pkl_dir):
        if not file.endswith(".pkl"):
            continue
        full_path = os.path.join(pkl_dir, file)
        try:
            with open(full_path, "rb") as f:
                data = pickle.load(f)

            df = data.get("per_center", None)
            if df is None or len(df) == 0:
                n_files_skipped += 1; continue

            df = df[df["state"] == state].copy()
            need = ["isi_entropy", rate_col]
            if df.empty or not set(need).issubset(df.columns):
                n_files_skipped += 1; continue

            df = df[need].replace([np.inf, -np.inf], np.nan).dropna()
            if df.empty:
                n_files_skipped += 1; continue

            # per-file z of d2
            d2_std = df["isi_entropy"].std(ddof=0)
            if not np.isfinite(d2_std) or d2_std == 0:
                n_files_skipped += 1; continue
            df["isi_entropy_norm"] = (df["isi_entropy"] - df["isi_entropy"].mean()) / d2_std

            # ---- FORCE 6 TILES (fallback to rank if qcut collapses) ----
            tiles = _force_sextiles(df[rate_col], k=n_bins)
            df["tile"] = tiles
            n_uniq = df["tile"].nunique(dropna=True)
            if n_uniq < min_bins_required:
                # not enough valid bins (e.g., <6 rows); skip file
                n_files_skipped += 1; continue

            df["file_id"] = os.path.splitext(file)[0]
            df["state"] = state
            df["rate_col"] = rate_col

            pooled.append(df[["file_id","state","rate_col","isi_entropy_norm",rate_col,"tile"]])
            n_files_used += 1

        except Exception:
            n_files_skipped += 1
            continue

    if not pooled:
        raise ValueError("No usable data loaded after per-file binning.")

    pooled_df = pd.concat(pooled, ignore_index=True)
    return pooled_df
def plot_isi_by_rate_tile_pooled_crossfileSEM(pooled_df, n_bins=6, title=None, color="tab:blue", save_path=None):
    """
    Plot mean z(D²) per tile with SEM computed ACROSS FILES.
    Steps:
      1) per-file, per-tile mean of d2_norm
      2) across-file mean and SEM of those per-file means
    """
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.stats import linregress

    # keep only 1..n_bins
    d = pooled_df.copy()
    d = d[d["tile"].between(1, n_bins)]

    # --- (1) per-file mean within each tile
    pf = (d.groupby(["file_id","tile"], observed=True)["isi_entropy_norm"]
            .mean()
            .rename("pf_mean")
            .reset_index())

    # --- (2) across-file mean & SEM per tile
    tile_mean = pf.groupby("tile")["pf_mean"].mean()
    tile_sem  = pf.groupby("tile")["pf_mean"].apply(lambda x: x.std(ddof=1) / np.sqrt(x.count()))
    n_files_per_tile = pf.groupby("tile")["file_id"].nunique()

    # ensure we have an entry for every 1..n_bins
    idx = pd.Index(range(1, n_bins+1), name="tile")
    tile_mean = tile_mean.reindex(idx)
    tile_sem  = tile_sem.reindex(idx)
    n_files_per_tile = n_files_per_tile.reindex(idx).fillna(0).astype(int)

    xs = tile_mean.index.to_numpy(int)
    ys = tile_mean.values
    es = tile_sem.values

    # linear trend on cross-file means
    ok = np.isfinite(ys)
    if ok.sum() >= 2:
        lr = linregress(xs[ok], ys[ok])
        r2 = lr.rvalue**2
        r_value=lr.rvalue
        p  = lr.pvalue
        p_str = ("p < 1e-45" if p < 1e-45 else (f"p = {p:.2e}" if p < 0.001 else f"p = {p:.3f}"))
    else:
        r2, p_str = np.nan, "p = NA"
    star_str=p_to_stars(p) if ok.sum() >= 2 else ""
    
    print("\n📊 Linear trend on cross-file means:")
    print(f"  {p_str}")
    print(f" r = {r_value:.3f}" if ok.sum() >= 2 else "r = NA")
    plt.figure(figsize=(5,4))
    plt.errorbar(xs, ys, yerr=es, fmt='-o', capsize=4, color=color, linewidth=2, label="mean ± SEM (across files)")

    # # optional OLS line for reference
    # if ok.sum() >= 2:
    #     yfit = lr.intercept + lr.slope * xs
    #     plt.plot(xs, yfit, linestyle="--", linewidth=1.5, color=color, alpha=0.6, label="OLS on cross-file means")

    plt.xticks(xs)
    plt.xlabel("Rate tiles (low → high)")
    plt.ylabel("ISI entropy (z)")
    try:
        import seaborn as sns
        sns.despine()
    except Exception:
        pass
    ax = plt.gca()
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Figure saved to {save_path}")
    plt.show()
# ---- data (../csv) ----
df_ripple_pooled_isi = read_csv('isi_entropy_by_swr_rate_sextile.csv')
# ---- the same from the recordings, with analysis_scripts/: ----
# pkl_dir=r"D:\Documents\GitHub\Sleep_criticaltiy\data\ripple_barr\ISI_entropy\ripple"
# df_ripple_pooled_isi = load_pool_rate_binned_isi(
#     pkl_dir, rate_col="ripple_rate", state="NREM", n_bins=6
# )
plot_isi_by_rate_tile_pooled_crossfileSEM(
    df_ripple_pooled_isi, n_bins=6,
    title="ISI entropy rate quantiles (low → high)",
    color="tab:red",
    save_path=r"../figure_output\fig2_H_isi_entropy_by_d2_tile_pooled.svg"
)